# Higher Order Functions in Databricks SQL

**Higher order functions** are functions that operate on complex data types (arrays, maps) by accepting **lambda functions** (anonymous functions) as arguments. They allow you to transform, filter, or aggregate array/map elements **without exploding** the data into multiple rows.

## Key Concept

Instead of:
1. Exploding array → Multiple rows
2. Transforming each element
3. Re-aggregating back to array

You can:
1. Apply a function **directly on the array** in a single operation

---

## Common Higher Order Functions

| Function | Purpose | Syntax |
|----------|---------|--------|
| **`transform()`** | Transform each element | `transform(array, x -> expression)` |
| **`filter()`** | Keep elements matching condition | `filter(array, x -> condition)` |
| **`exists()`** | Check if any element matches | `exists(array, x -> condition)` |
| **`aggregate()`** | Custom aggregation | `aggregate(array, init, merge, finish)` |
| **`reduce()`** | Reduce to single value | `reduce(array, init, (acc, x) -> expr)` |
| **`zip_with()`** | Combine two arrays | `zip_with(array1, array2, (x, y) -> expr)` |
| **`array_sort()`** | Custom sorting | `array_sort(array, (x, y) -> comparison)` |
| **`forall()`** | Check if all elements match | `forall(array, x -> condition)` |

---

## Advantages Over Explode

✅ **No row explosion** — keeps data at original granularity  
✅ **Better performance** — single pass, no shuffle  
✅ **Cleaner code** — no GROUP BY needed  
✅ **Preserves array structure** — maintains element relationships  

Let's explore each function with examples!

In [0]:
%sql
-- Create sample data for demonstrations
CREATE OR REPLACE TEMP VIEW orders AS
SELECT 1 as order_id, 'Alice' as customer, array(10.50, 25.00, 15.75, 8.25) as prices, array('book', 'mouse', 'cable', 'pen') as items
UNION ALL
SELECT 2 as order_id, 'Bob' as customer, array(100.00, 50.00) as prices, array('keyboard', 'headphones') as items
UNION ALL
SELECT 3 as order_id, 'Charlie' as customer, array(5.00, 3.50, 2.00) as prices, array('eraser', 'pencil', 'sharpener') as items;

SELECT * FROM orders;

In [0]:
%sql
-- TRANSFORM: Apply a function to each element in an array
-- Syntax: transform(array, element -> expression)

-- Example 1: Add 10% tax to all prices
SELECT 
  order_id,
  customer,
  prices as original_prices,
  transform(prices, x -> x * 1.10) as prices_with_tax,
  transform(prices, x -> round(x * 1.10, 2)) as prices_rounded
FROM orders;

-- Example 2: Convert item names to uppercase
SELECT 
  order_id,
  customer,
  items,
  transform(items, x -> upper(x)) as items_upper
FROM orders;

In [0]:
%sql
-- Comparing TRANSFORM (higher order) vs EXPLODE approach

-- Method 1: Using EXPLODE (old way - creates intermediate rows)
WITH exploded AS (
  SELECT 
    order_id,
    customer,
    explode(prices) as price
  FROM orders
),
aggregated AS (
  SELECT 
    order_id,
    customer,
    collect_list(price * 1.10) as prices_with_tax
  FROM exploded
  GROUP BY order_id, customer
)
SELECT * FROM aggregated;

-- Method 2: Using TRANSFORM (new way - single operation)
SELECT 
  order_id,
  customer,
  transform(prices, x -> x * 1.10) as prices_with_tax
FROM orders;

-- Result: Same output, but TRANSFORM is cleaner and more efficient!

In [0]:
%sql
-- FILTER: Keep only elements that match a condition
-- Syntax: filter(array, element -> boolean_condition)

-- Example 1: Keep only prices above $10
SELECT 
  order_id,
  customer,
  prices,
  filter(prices, x -> x > 10) as high_value_prices,
  size(filter(prices, x -> x > 10)) as high_value_count
FROM orders;

-- Example 2: Keep only items containing 'e'
SELECT 
  order_id,
  customer,
  items,
  filter(items, x -> x like '%e%') as items_with_e
FROM orders;

-- Example 3: Multiple conditions
SELECT 
  order_id,
  customer,
  prices,
  filter(prices, x -> x >= 5 AND x <= 50) as mid_range_prices
FROM orders;

In [0]:
%sql
-- EXISTS: Returns true if at least one element matches the condition
-- Syntax: exists(array, element -> boolean_condition)

-- Example 1: Check if order has any expensive items (> $50)
SELECT 
  order_id,
  customer,
  prices,
  exists(prices, x -> x > 50) as has_expensive_item,
  exists(prices, x -> x < 5) as has_cheap_item
FROM orders;

-- Example 2: Check if specific item exists
SELECT 
  order_id,
  customer,
  items,
  exists(items, x -> x = 'book') as has_book,
  exists(items, x -> x like '%phone%') as has_phone_item
FROM orders;

In [0]:
%sql
-- FORALL: Returns true if ALL elements match the condition
-- Syntax: forall(array, element -> boolean_condition)

-- Example 1: Check if all prices are above $5
SELECT 
  order_id,
  customer,
  prices,
  forall(prices, x -> x > 5) as all_above_5,
  forall(prices, x -> x > 1) as all_above_1
FROM orders;

-- Example 2: Check if all items are lowercase
SELECT 
  order_id,
  customer,
  items,
  forall(items, x -> x = lower(x)) as all_lowercase
FROM orders;

In [0]:
%sql
-- AGGREGATE: Custom aggregation logic on array elements
-- Syntax: aggregate(array, initial_value, merge_function, finish_function)
-- Parameters:
--   1. array: input array
--   2. initial_value: starting accumulator value
--   3. merge_function: (accumulator, element) -> new_accumulator
--   4. finish_function: final transformation (optional)

-- Example 1: Sum all prices
SELECT 
  order_id,
  customer,
  prices,
  aggregate(prices, CAST(0.0 AS DOUBLE), (acc, x) -> acc + x) as total_price
FROM orders;

-- Example 2: Calculate average using aggregate
SELECT 
  order_id,
  customer,
  prices,
  aggregate(
    prices,
    named_struct('sum', CAST(0.0 AS DOUBLE), 'count', 0),
    (acc, x) -> named_struct('sum', acc.sum + x, 'count', acc.count + 1),
    acc -> acc.sum / acc.count
  ) as avg_price
FROM orders;

-- Example 3: Find maximum price
SELECT 
  order_id,
  customer,
  prices,
  aggregate(prices, CAST(0.0 AS DOUBLE), (acc, x) -> CASE WHEN x > acc THEN x ELSE acc END) as max_price
FROM orders;

In [0]:
%sql
SELECT 
  order_id,
  customer,
  prices,
  aggregate(prices, CAST(0.0 AS DOUBLE), (acc, x) -> acc + x) as total_price
FROM orders;

In [0]:
%sql
-- REDUCE: Simplified version of aggregate (no finish function)
-- Syntax: reduce(array, initial_value, (accumulator, element) -> new_accumulator)

-- Example 1: Sum all prices
SELECT 
  order_id,
  customer,
  prices,
  reduce(prices, CAST(0.0 AS DOUBLE), (acc, x) -> acc + x) as total_price
FROM orders;

-- Example 2: Concatenate all items
SELECT 
  order_id,
  customer,
  items,
  reduce(items, '', (acc, x) -> concat(acc, CASE WHEN acc = '' THEN '' ELSE ', ' END, x)) as items_list
FROM orders;

-- Example 3: Count items above $10
SELECT 
  order_id,
  customer,
  prices,
  reduce(prices, 0, (acc, x) -> acc + CASE WHEN x > 10 THEN 1 ELSE 0 END) as expensive_count
FROM orders;

In [0]:
%sql
-- ZIP_WITH: Combine two arrays element-wise
-- Syntax: zip_with(array1, array2, (element1, element2) -> expression)

-- Example 1: Calculate total price (quantity × price)
SELECT 
  order_id,
  customer,
  items,
  prices,
  zip_with(items, prices, (item, price) -> concat(item, ': $', cast(price as string))) as item_prices
FROM orders;

-- Example 2: Element-wise addition
SELECT 
  zip_with(array(1, 2, 3), array(10, 20, 30), (x, y) -> x + y) as sums,
  zip_with(array(10, 20, 30), array(5, 10, 15), (x, y) -> x - y) as differences,
  zip_with(array(2, 4, 6), array(3, 3, 3), (x, y) -> x * y) as products;

-- Example 3: Create struct from two arrays
SELECT 
  order_id,
  zip_with(items, prices, (item, price) -> named_struct('product', item, 'cost', price)) as product_details
FROM orders;

In [0]:
%sql
-- ARRAY_SORT: Sort array with custom comparator
-- Syntax: array_sort(array, (element1, element2) -> comparison)
-- Return: negative (x < y), zero (x = y), positive (x > y)

-- Example 1: Sort prices in descending order
SELECT 
  order_id,
  customer,
  prices,
  array_sort(prices) as prices_asc,
  array_sort(prices, (x, y) -> CASE WHEN y < x THEN -1 WHEN y > x THEN 1 ELSE 0 END) as prices_desc
FROM orders;

-- Example 2: Sort items by length
SELECT 
  order_id,
  customer,
  items,
  array_sort(items, (x, y) -> length(x) - length(y)) as items_by_length
FROM orders;

-- Example 3: Sort structs by price
CREATE OR REPLACE TEMP VIEW products AS
SELECT 
  1 as order_id,
  array(
    named_struct('name', 'Laptop', 'price', 1200),
    named_struct('name', 'Mouse', 'price', 25),
    named_struct('name', 'Keyboard', 'price', 75)
  ) as products;

SELECT 
  order_id,
  array_sort(products, (x, y) -> CAST(x.price - y.price as INT)) as products_sorted_by_price
FROM products;

In [0]:
%sql
-- You can chain multiple higher order functions for complex transformations

-- Example 1: Filter, transform, then sort
SELECT 
  order_id,
  customer,
  prices,
  -- Step 1: Filter prices > $5
  -- Step 2: Add 10% tax
  -- Step 3: Sort descending
  array_sort(
    transform(
      filter(prices, x -> x > 5),
      x -> round(x * 1.10, 2)
    ),
    (x, y) -> CASE WHEN y < x THEN -1 WHEN y > x THEN 1 ELSE 0 END
  ) as processed_prices
FROM orders;

-- Example 2: Combine multiple operations
SELECT 
  order_id,
  customer,
  items,
  prices,
  -- Filter high-value items and create formatted strings
  transform(
    zip_with(
      filter(items, (x, i) -> prices[i] > 10),
      filter(prices, x -> x > 10),
      (item, price) -> concat(upper(item), ': $', cast(price as string))
    ),
    x -> x
  ) as high_value_items
FROM orders;

In [0]:
%sql
-- Real-world practical examples

-- Use Case 1: Data Quality - Remove NULL values
CREATE OR REPLACE TEMP VIEW data_with_nulls AS
SELECT 1 as id, array(10, NULL, 20, NULL, 30) as values;

SELECT 
  id,
  values,
  filter(values, x -> x IS NOT NULL) as cleaned_values
FROM data_with_nulls;

-- Use Case 2: Feature Engineering - Bin numerical values
SELECT 
  order_id,
  customer,
  prices,
  transform(prices, x -> 
    CASE 
      WHEN x < 10 THEN 'low'
      WHEN x < 50 THEN 'medium'
      ELSE 'high'
    END
  ) as price_categories
FROM orders;

-- Use Case 3: String Processing - Normalize text
SELECT 
  order_id,
  customer,
  items,
  transform(items, x -> trim(lower(x))) as normalized_items
FROM orders;

-- Use Case 4: Calculate discounts based on quantity
CREATE OR REPLACE TEMP VIEW bulk_orders AS
SELECT 
  1 as order_id,
  array(1, 5, 10, 20) as quantities,
  array(100.0, 100.0, 100.0, 100.0) as unit_prices;

SELECT 
  order_id,
  quantities,
  unit_prices,
  zip_with(quantities, unit_prices, (qty, price) -> 
    price * qty * CASE 
      WHEN qty >= 10 THEN 0.9  -- 10% discount
      WHEN qty >= 5 THEN 0.95   -- 5% discount
      ELSE 1.0                  -- No discount
    END
  ) as total_prices_with_discount
FROM bulk_orders;

## Higher Order Functions - Complete Reference

---

### Function Comparison

| Function | Purpose | Syntax | Returns |
|----------|---------|--------|----------|
| **`transform()`** | Transform each element | `transform(array, x -> expr)` | Array |
| **`filter()`** | Keep matching elements | `filter(array, x -> condition)` | Array |
| **`exists()`** | Check if any match | `exists(array, x -> condition)` | Boolean |
| **`forall()`** | Check if all match | `forall(array, x -> condition)` | Boolean |
| **`aggregate()`** | Custom aggregation | `aggregate(array, init, merge, finish)` | Any type |
| **`reduce()`** | Simplified aggregation | `reduce(array, init, (acc, x) -> expr)` | Any type |
| **`zip_with()`** | Combine two arrays | `zip_with(arr1, arr2, (x, y) -> expr)` | Array |
| **`array_sort()`** | Custom sorting | `array_sort(array, (x, y) -> comparison)` | Array |

---

### Lambda Function Syntax

Higher order functions use **lambda expressions** (anonymous functions):

```sql
-- Single parameter
x -> x * 2

-- Two parameters
(x, y) -> x + y

-- Complex expression
x -> CASE WHEN x > 10 THEN 'high' ELSE 'low' END

-- Multiple statements (use parentheses)
x -> (x * 1.1)
```

---

### Common Patterns

#### 1. Transform + Filter (Clean and Modify)
```sql
SELECT transform(
  filter(prices, x -> x IS NOT NULL AND x > 0),
  x -> round(x * 1.1, 2)
) as valid_prices_with_tax
FROM orders;
```

#### 2. Exists + Filter (Conditional Processing)
```sql
SELECT 
  CASE 
    WHEN exists(prices, x -> x > 100) 
    THEN filter(prices, x -> x > 100)
    ELSE prices
  END as filtered_prices
FROM orders;
```

#### 3. Aggregate + Transform (Calculate and Format)
```sql
SELECT 
  transform(
    prices,
    x -> concat('$', cast(x as string))
  ) as formatted_prices,
  aggregate(prices, 0.0, (acc, x) -> acc + x) as total
FROM orders;
```

#### 4. Zip_With + Reduce (Combine and Aggregate)
```sql
SELECT 
  reduce(
    zip_with(quantities, prices, (q, p) -> q * p),
    0.0,
    (acc, x) -> acc + x
  ) as order_total
FROM orders;
```

---

### Performance Comparison

| Approach | Rows Created | Shuffles | Performance |
|----------|-------------|----------|-------------|
| **Higher Order Functions** | No explosion | 0 | ✅ Fast |
| **Explode + GROUP BY** | N × array_size | 1-2 | ⚠️ Slower |

#### Example: Transform 1M rows with 10-element arrays

* **Higher Order**: 1M rows processed → 1M rows output
* **Explode**: 1M rows → 10M rows → shuffle → 1M rows output

**Result**: Higher order functions are typically **2-10x faster** for array operations.

---

### When to Use Each Function

#### TRANSFORM
✅ Apply same operation to all elements  
✅ Type conversion (string to int, etc.)  
✅ Mathematical operations  
✅ String manipulation  

#### FILTER
✅ Remove NULL/invalid values  
✅ Keep elements meeting criteria  
✅ Data quality checks  
✅ Subset extraction  

#### EXISTS / FORALL
✅ Boolean checks without exploding  
✅ WHERE clause conditions on arrays  
✅ Validation logic  
✅ Feature flags  

#### AGGREGATE / REDUCE
✅ Custom aggregations (sum, avg, max, min)  
✅ String concatenation  
✅ Complex calculations  
✅ When built-in functions aren't enough  

#### ZIP_WITH
✅ Element-wise operations on two arrays  
✅ Combining related arrays  
✅ Calculating totals (quantity × price)  
✅ Creating structs from parallel arrays  

#### ARRAY_SORT
✅ Custom sorting logic  
✅ Sort by struct fields  
✅ Sort by computed values  
✅ Descending order  

---

### Best Practices

✅ **Use higher order functions instead of explode when possible** — Better performance  
✅ **Chain functions for complex transformations** — Still more efficient than explode  
✅ **Keep lambda expressions simple** — Complex logic can be hard to debug  
✅ **Use EXISTS for existence checks** — Don't explode just to check if value exists  
✅ **Prefer REDUCE over AGGREGATE for simple cases** — Cleaner syntax  

❌ **Don't use higher order functions when you need multiple rows** — Use explode for that  
❌ **Don't nest too deeply** — Can impact readability  
❌ **Don't use AGGREGATE when built-in functions exist** — Use array_sum, array_max, etc. if available  

---

### PySpark Equivalent

For PySpark users, here's the mapping:

| SQL | PySpark |
|-----|----------|
| `transform(arr, x -> x * 2)` | `transform(col("arr"), lambda x: x * 2)` |
| `filter(arr, x -> x > 10)` | `filter(col("arr"), lambda x: x > 10)` |
| `exists(arr, x -> x > 10)` | `exists(col("arr"), lambda x: x > 10)` |
| `aggregate(arr, 0, (a,x)->a+x)` | `aggregate(col("arr"), lit(0), lambda a,x: a+x)` |
| `zip_with(a, b, (x,y)->x+y)` | `zip_with(col("a"), col("b"), lambda x,y: x+y)` |

---

### Additional Resources

* **Databricks Documentation**: Search for "higher order functions" in Databricks docs
* **Performance Tuning**: Monitor query plans to see shuffle reduction
* **Complex Examples**: Check Databricks blogs for advanced patterns